# Standardize Charges

Apply the standardization pipeline from `standardize_charges.py` to each individual charge in checkpoint13.
Produces a `standardized_charges` column with canonical charge names (semicolon-delimited).
Drops the 51 expanded `charge_N` and `statute_N` columns, collapsing them into concise summary columns.

**Input:** `checkpoint13_cleaned_charges.csv` (69 columns)
**Output:** `checkpoint14_standardized_charges.csv` (~20 columns)

## NEW!! This file is no longer relevant because clean_charges.ipnyb no longer splits the charges , run the code below to just get cp13 to 14 to get to misdemeanor step

In [1]:
import os
import pandas as pd

# Paths
NOTEBOOK_DIR = os.getcwd()

DATA_DIR = os.path.join(
    NOTEBOOK_DIR,
    "..",
    "data",
    "missing_dates_csv",
)

CHECKPOINTS_DIR = os.path.join(DATA_DIR, "md_checkpoints")

# Input: checkpoint 13
DATA_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint13_cleaned_charges.csv",
)

# Output: checkpoint 14
OUT_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint14_standardized_charges.csv",
)

# Load checkpoint 13
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df):,} rows")
print(f"Columns: {len(df.columns)}")

# (Run your standardization code here)

# Save checkpoint 14
df.to_csv(OUT_PATH, index=False)

print(f"\nSaved standardized checkpoint to:")
print(os.path.abspath(OUT_PATH))

Loaded 428,527 rows
Columns: 19

Saved standardized checkpoint to:
/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint14_standardized_charges.csv


In [ ]:
import os
import pandas as pd

# Paths
NOTEBOOK_DIR = os.getcwd()

DATA_DIR = os.path.join(
    NOTEBOOK_DIR,
    "..",
    "data",
    "missing_dates_csv",
)

CHECKPOINTS_DIR = os.path.join(DATA_DIR, "md_checkpoints")

# Input: checkpoint 13
DATA_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint13_cleaned_charges.csv",
)

# Output: checkpoint 14
OUT_PATH = os.path.join(
    CHECKPOINTS_DIR,
    "checkpoint14_standardized_charges.csv",
)

# Load checkpoint 13
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df):,} rows")
print(f"Columns: {len(df.columns)}")

# (Run your standardization code here)

# Save checkpoint 14
df.to_csv(OUT_PATH, index=False)

print(f"\nSaved standardized checkpoint to:")
print(os.path.abspath(OUT_PATH))

### Imports & Paths

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from standardize_charges import (
    extract_warrant_type,
    strip_statute_refs,
    clean_charge,
    extract_offense_number,
    normalize_charge,
)

NOTEBOOK_DIR = os.getcwd()
DATA_DIR = os.path.join(NOTEBOOK_DIR, "..", "..", "data")
CHECKPOINTS_DIR = os.path.join(DATA_DIR, "checkpoints")

DATA_PATH = os.path.join(CHECKPOINTS_DIR, "checkpoint13_cleaned_charges.csv")
OUT_PATH = os.path.join(CHECKPOINTS_DIR, "checkpoint14_standardized_charges.csv")

### Load checkpoint13

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Rows with charges: {df['cleaned_charges'].notna().sum()}")
print(f"Rows without charges: {df['cleaned_charges'].isna().sum()}")
df.head()

Shape: (424273, 69)
Columns: 69
Rows with charges: 4542
Rows without charges: 419731


,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,latitude,longitude,Cleaned Location,...,statute_14,statute_15,statute_16,statute_17,statute_18,statute_19,statute_20,statute_21,statute_22,statute_23
0,2018-01-01 00:01:00,NOISE ORD,3 HARRIMAN ST,Yes,NaN,1974-07-03,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,42.718522,-71.148148,3 HARRIMAN ST,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-01 00:08:00,LOUD NOISE,1 HARRIMAN ST FL 2,Yes,NaN,1979-01-21,A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PA...,42.718522,-71.148148,1 HARRIMAN ST,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-01-01 00:11:00,ALARM/BURG,16 ALLEN ST,No,MATOS,NaN,NaN,42.710782,-71.151911,16 ALLEN ST,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-01-01 00:14:00,DISORDERLY,11 SUMMER ST,No,NaN,NaN,NaN,42.711117,-71.153015,11 SUMMER ST,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-01-01 00:27:00,EXTRA SURVEIL,57 SPRINGFIELD ST,Yes,WARD SIX CLUB,2002-06-26,A&B DOMESTIC NO 209A IN EFFECT,42.699339,-71.156938,57 SPRINGFIELD ST,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Define standardization function
Wraps the full pipeline from `standardize_charges.py` for a single charge string.

In [3]:
def standardize_single_charge(raw):
    """Run the full standardization pipeline on one charge string.
    
    Returns the canonical base_charge name, or None if input is empty.
    """
    if pd.isna(raw) or not str(raw).strip():
        return None

    raw = str(raw).strip()

    # Step 1: Strip warrant prefix
    _, remaining = extract_warrant_type(raw)

    # Step 2: Strip statute references
    cleaned = strip_statute_refs(remaining)

    # Step 3: Normalize whitespace and trailing artifacts
    base = clean_charge(cleaned)

    # Step 4: Strip offense number suffix
    _, base = extract_offense_number(base)

    # Step 5: Map to canonical form
    base = normalize_charge(base)

    return base if base else None


# Quick test
test_cases = [
    "default warrant: license suspended, op mv with",
    "a&b on family / household member / intimate partne",
    "standard warrant: destruction of prop -$1200, malicious c 266",
    "TRESPASS c266 S120",
]
for tc in test_cases:
    print(f"  {tc}")
    print(f"    -> {standardize_single_charge(tc)}")
    print()

  default warrant: license suspended, op mv with
    -> license suspended, op mv with

  a&b on family / household member / intimate partne
    -> a&b on family / household member

  standard warrant: destruction of prop -$1200, malicious c 266
    -> destruction of prop -$1200, malicious

  TRESPASS c266 S120
    -> trespass c266 s120



### Build `standardized_charges` column
Apply the standardization function to each `charge_N` column, then join results into a single semicolon-delimited string.

In [4]:
charge_cols = [c for c in df.columns if re.match(r"^charge_\d+$", c)]
print(f"Found {len(charge_cols)} charge columns: {charge_cols[0]} ... {charge_cols[-1]}")

# Standardize each charge_N column into a temporary list of Series
std_series = []
for col in charge_cols:
    std_series.append(df[col].apply(standardize_single_charge))

# Build the standardized_charges column by joining non-null values per row
def join_standardized(row_values):
    parts = [v for v in row_values if v is not None]
    return "; ".join(parts) if parts else np.nan

std_df = pd.DataFrame({f"std_{i}": s for i, s in enumerate(std_series)})
df["standardized_charges"] = std_df.apply(lambda row: join_standardized(row.values), axis=1)

print(f"Rows with standardized_charges: {df['standardized_charges'].notna().sum()}")
print(f"Rows without: {df['standardized_charges'].isna().sum()}")

Found 28 charge columns: charge_1 ... charge_28
Rows with standardized_charges: 4542
Rows without: 419731


### Drop expanded columns
Remove the 51 intermediate `charge_N`, `statute_N`, and `cleaned_charges` columns.

In [5]:
cols_before = len(df.columns)

# Identify all expanded columns to drop
charge_cols = [c for c in df.columns if re.match(r"^charge_\d+$", c)]
statute_cols = [c for c in df.columns if re.match(r"^statute_\d+$", c)]
cols_to_drop = charge_cols + statute_cols + ["cleaned_charges"]

df = df.drop(columns=cols_to_drop, errors="ignore")

print(f"Dropped {cols_before - len(df.columns)} columns ({cols_before} -> {len(df.columns)})")
print(f"\nRemaining columns:")
for c in df.columns:
    print(f"  {c}")

Dropped 52 columns (70 -> 18)

Remaining columns:
  Date
  Type
  Location
  Arrested
  Location Prefix
  DOB
  Charges
  latitude
  longitude
  Cleaned Location
  person_id
  category_archive
  Year
  crime_severity
  category
  Age
  statutes
  standardized_charges


### Verify

In [6]:
# Count check
std_count = df["standardized_charges"].notna().sum()
print(f"Rows with standardized_charges: {std_count}")
print(f"Expected: 4542")
print(f"Match: {std_count == 4542}")

# Unique standardized charges
all_std = df["standardized_charges"].dropna().str.split("; ").explode()
print(f"\nTotal individual charge instances: {len(all_std)}")
print(f"Unique standardized charges: {all_std.nunique()}")
print(f"Expected: <= 394")

# Spot-check some rows
print("\nSample rows with charges:")
sample = df[df["standardized_charges"].notna()].head(5)
for _, row in sample.iterrows():
    print(f"  Original:      {str(row['Charges'])[:100]}")
    print(f"  Standardized:  {row['standardized_charges']}")
    print()

# Final shape
print(f"Final shape: {df.shape}")

Rows with standardized_charges: 4542
Expected: 4542
Match: True

Total individual charge instances: 12308
Unique standardized charges: 394
Expected: <= 394

Sample rows with charges:
  Original:      A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PARTNE; STRANGULATION OR SUFFOCATION
  Standardized:  a&b on family / household member; strangulation or suffocation

  Original:      A&B ON FAMILY / HOUSEHOLD MEMBER / INTIMATE PARTNE
  Standardized:  a&b on family / household member

  Original:      A&B DOMESTIC NO 209A IN EFFECT
  Standardized:  a&b domestic no 209a in effect

  Original:      USE MV WITHOUT AUTHORITY c90 S24; LARCENY UNDER $250 c266 S30
  Standardized:  use mv without authority; larceny under $250

  Original:      WITNESS, INTIMIDATE c268 S13B; FALSE NAME/SS# TO LAW ENFORCEMENT; THREAT TO COMMIT CRIME c275 S2; AS
  Standardized:  witness, intimidate; false name/ss# to law enforcement; threat to commit crime; assault w/dangerous weapon; kidnapping; a&b on family / household

### Save checkpoint14

In [7]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved checkpoint14: {OUT_PATH}")
print(f"Shape: {df.shape}")

Saved checkpoint14: c:\Users\Indel\Documents\gatewayinitiative-lawrencepd\scripts\..\data\checkpoints\checkpoint14_standardized_charges.csv
Shape: (424273, 18)
